# LDPC码通用执行过程：编码与迭代译码的标准步骤

在不失一般性的前提下，以下是教材中关于一个参数为 $(N, K)$、校验矩阵为 $M \times N$ 的 LDPC 码的完整系统执行流程序列。

## 阶段一：系统初始化
1. **构造校验矩阵**：根据设计准则（如度分布多项式或有限几何）生成稀疏的 $M \times N$ 奇偶校验矩阵 $H$。
2. **推导生成矩阵**：为消除代数运算的歧义，通过引入显式的变换矩阵将原矩阵转化为系统码形式。
   * **满秩假设**：假设 $H$ 为满行秩矩阵，即 $\text{rank}(H) = M$，此时 $K = N - M$。（注：若生成的 $H$ 存在线性相关行，即冗余校验方程，需预先剔除，保证其满秩后再进行后续推导）。
   * **列置换与基底选择**：由于 $\text{rank}(H) = M$，必然可以在 $H$ 中找到 $M$ 个线性无关的列。定义 $N \times N$ 的正交置换矩阵 $\Pi$ 将这 $M$ 个线性无关的列置换到矩阵的最右侧。此时有 $H\Pi = [A_{M \times K} \mid B_{M \times M}]$，其中右侧的方阵 $B$ 由线性无关列组成，必定满秩且可逆。
   * **行变换化零**：定义 $M \times M$ 的可逆满秩矩阵 $T = B^{-1}$ 表示所有初等行变换的累积。
   * 将原校验矩阵 $H$ 转换为**系统校验矩阵** $H_{sys}$（与原矩阵 $H$ 严格区分）：$H_{sys} = T H \Pi = B^{-1} [A \mid B] = [B^{-1}A \mid I_{M \times M}] = [P_{M \times K} \mid I_{M \times M}]$ *(其中 $I_{M \times M}$ 为 $M$ 阶单位阵，$P_{M \times K} = B^{-1}A$ 为非系统部分的稠密矩阵)*
   * 构造对应的**系统生成矩阵** $G_{sys}$：$G_{sys} = [I_{K \times K} \mid P^T_{K \times M}]$。*(正交性证明：在 $\text{GF}(2)$ 域中，执行分块矩阵乘法可得 $G_{sys}H_{sys}^T = [I_{K \times K} \mid P^T] [P^T \mid I_{M \times M}]^T = I_{K \times K}P^T + P^T I_{M \times M} = P^T + P^T = \mathbf{0} \pmod 2$)*
   * 恢复原始变量顺序得到最终的**生成矩阵** $G$：由于列置换矩阵满足 $\Pi^{-1} = \Pi^T$，定义 $G = G_{sys} \Pi^T$。该 $G$ 严格满足 $GH^T = \mathbf{0}$。

## 阶段二：编码执行过程 (Encoding)
**输入**：长度为 $K$ 的原始信息比特向量 $\mathbf{u} = [u_1, u_2, \dots, u_K]$。
**执行步骤**：
1. 执行矩阵乘法运算：$\mathbf{c} = \mathbf{u}G \pmod 2$。
2. 得到长度为 $N$ 的码字向量 $\mathbf{c} = [c_1, c_2, \dots, c_N]$。由于可能经历了列置换 $\Pi$，码字的系统位可能不再严格位于前 $K$ 位，而是分散在 $N$ 位中。

## 阶段二（扩展）：稀疏校验矩阵的工程困境与主流高效编码方案

如果按照上述“阶段一”的逻辑，完全随机地构造一个满秩的稀疏校验矩阵 $H$，并在“阶段二”使用 $\mathbf{c} = \mathbf{u}G$ 进行直接编码，会面临极其严重的工程瓶颈：**稠密生成矩阵问题**。

尽管 $H$ 是极度稀疏的，但通过高斯消元法求逆（$T=B^{-1}$）得到的非系统部分矩阵 $P = B^{-1}A$ 往往是**极其稠密**的。这意味着生成的 $G_{sys}$ 也是稠密的。此时，计算 $\mathbf{c} = \mathbf{u}G$ 的编码复杂度为 $\mathcal{O}(K \times (N-K))$，即 $\mathcal{O}(N^2)$。对于动辄几万甚至几十万码长的现代通信系统，这种平方级的编码复杂度在芯片上是不可接受的。因此，现代标准不再使用随机稀疏矩阵，而是人为设计具有特殊结构的 $H$ 以实现线性时间 $\mathcal{O}(N)$ 编码。

### 方案一：Richardson-Urbanke (RU) 算法 (近似线性时间编码)
**H矩阵结构设计**：不再强制化为严格的系统形式 $[P \mid I]$，而是通过纯粹的行列置换矩阵 $P_{row}$ 和 $P_{col}$，将原稀疏矩阵 $H$ 转化为近似下三角形式（Approximate Lower Triangular, ALT）的新矩阵 $H_{ALT}$：
$H_{ALT} = P_{row} H P_{col} = \begin{bmatrix} A_{(M-g) \times K} & B_{(M-g) \times g} & T_{(M-g) \times (M-g)} \\ C_{g \times K} & D_{g \times g} & E_{g \times (M-g)} \end{bmatrix}$
*(注：设码字长度为 $N$，信息位长度为 $K$，校验位长度为 $M = N - K$。$g$ 为间隙部分大小。其中 $T$ 为极度稀疏的下三角方阵。)*

**如何化为下三角阵 $T$ (严格的 RU 贪心三角化算法)**：
为了保持矩阵极度稀疏，该步骤**仅允许纯粹的行列置换**。由于 $T$ 是**下三角矩阵**（主对角线右上方全为 0），我们必须通过不断寻找“残余行重为 1 的行”来从上到下构建它。严格执行步骤如下：
1. **寻找行重为 1 的行 (Find Residual Row Weight 1)**：在当前尚未处理的矩阵区域（初始为整个 $H$ 矩阵）中，扫描寻找只包含一个“1”的行。
2. **对角线安置 (Diagonal Placement)**：将找到的这行交换到当前处理区域的第一行，并将那个唯一的“1”所在的列交换到第一列。由于该行在当前区域内只有一个 1，所以该“1”右侧的元素必定全为 0。这严格满足了下三角矩阵第一行的结构要求（此时这个 1 成为 $T$ 的对角线元素）。
3. **降阶与连锁反应 (Reduction & Avalanche)**：将这第一行和第一列从“未处理区域”中剥离（它们已正式确认属于 $T$ 矩阵的成分）。**核心机制**：由于我们从系统中划去了一整列，剩余未处理矩阵中与该列相交的行的 1 的个数会随之减少（即残余行重降低）。这种降低会触发“连锁反应”，暴露出新的行重为 1 的行。回到步骤 1 循环执行。
4. **卡壳与间隙生成 (Stagnation & Gap Creation)**：如果某一步发现当前未处理区域没有任何行重为 1 的行了（即剩余所有行的残余行重均 $\ge 2$），贪心过程“卡壳”。此时必须妥协：在未处理的列中主动挑出一列（通常启发式地选择列重最大的列），**将其彻底踢出后续的下三角构造过程，划归到间隙（Gap）区域**（即这列最终会变成左侧矩阵 $B$ 和 $D$ 的一部分）。剔除这列后，剩余相关行的残余行重会强制减小，从而极大概率重新产生行重为 1 的行，解开卡壳状态。每一次这样的“妥协踢出”操作，都会使得最终的间隙大小 $g$ 增加 1。反复执行直到划分完成。

**关于间隙大小 $g$ 的理论前提与上界**：
需要特别澄清：RU 论文**并没有**证明“任何稀疏满秩矩阵都能做到 $g = \mathcal{O}(\sqrt{N})$”。实际上，如果人为构造一个拓扑结构极其恶劣（例如内部高度连通、无法通过降阶触发连锁反应）的稀疏矩阵，$g$ 完全可能退化到与 $N$ 同阶的 $\mathcal{O}(N)$。
RU 定理是一个**基于概率系综（Ensemble）的渐近结果**。其严格表述为：对于由一对特定的、经过优化的度分布多项式 $(\lambda, \rho)$ 随机生成的 LDPC 码族，当满足一定设计条件时，随着码长 $N \to \infty$，贪心算法陷入卡壳的概率极低，平均间隙大小 $g$ 的期望值为 $\mathcal{O}(\sqrt{N})$。换言之，是**“精心设计的度分布特性”加上“稀疏性”**共同保证了 $g \ll N$ 的结果。在工程实践中（例如 $N=10^4 \sim 10^5$），对于优化好的随机 LDPC 码，$g$ 通常只有数十的量级。此时唯一涉及稠密矩阵乘法的复杂度 $g^2 \approx \mathcal{O}(N)$，完美保证了总编码复杂度为近似线性时间 $\mathcal{O}(N)$。

**RU编码执行流程与公式推导**：
将对应于新矩阵 $H_{ALT}$ 的置换后码字向量 $\mathbf{c}_{ALT}$ 划分为三部分：$\mathbf{c}_{ALT} = [\mathbf{u}, \mathbf{p}_1, \mathbf{p}_2]$，其中 $\mathbf{u}$ 为系统信息位（长度 $K$），$\mathbf{p}_1$ 为第一部分校验位（长度 $g$），$\mathbf{p}_2$ 为第二部分校验位（长度 $M-g$）。

**有效性证明（为什么这样定义满足编码需求）**：
为了使 $\mathbf{c}_{ALT}$ 成为合法的码字，必须满足核心校验方程 $H_{ALT} \mathbf{c}_{ALT}^T = \mathbf{0}$。展开该分块矩阵乘法：
$\begin{bmatrix} A & B & T \\ C & D & E \end{bmatrix} \begin{bmatrix} \mathbf{u}^T \\ \mathbf{p}_1^T \\ \mathbf{p}_2^T \end{bmatrix} = \begin{bmatrix} \mathbf{0} \\ \mathbf{0} \end{bmatrix}$
这等价于求解以下两个联立的矩阵方程组：
(1) $A\mathbf{u}^T + B\mathbf{p}_1^T + T\mathbf{p}_2^T = \mathbf{0}$
(2) $C\mathbf{u}^T + D\mathbf{p}_1^T + E\mathbf{p}_2^T = \mathbf{0}$

由方程 (1)，由于 $T$ 是下三角满秩方阵，可直接用前向代入法解出 $\mathbf{p}_2^T$：
$T\mathbf{p}_2^T = -A\mathbf{u}^T - B\mathbf{p}_1^T \implies \mathbf{p}_2^T = -T^{-1}(A\mathbf{u}^T + B\mathbf{p}_1^T)$

将 $\mathbf{p}_2^T$ 的表达式代入方程 (2) 中以消去 $\mathbf{p}_2$：
$C\mathbf{u}^T + D\mathbf{p}_1^T - E T^{-1}(A\mathbf{u}^T + B\mathbf{p}_1^T) = \mathbf{0}$

合并 $\mathbf{u}^T$ 与 $\mathbf{p}_1^T$ 的同类项：
$(-E T^{-1}A + C)\mathbf{u}^T + (-E T^{-1}B + D)\mathbf{p}_1^T = \mathbf{0}$

此时，定义核心稠密矩阵 $\Phi = -E T^{-1}B + D$，上式即化简为：
$\Phi \mathbf{p}_1^T = -(-E T^{-1}A + C)\mathbf{u}^T \implies \mathbf{p}_1^T = -\Phi^{-1} (-E T^{-1}A + C)\mathbf{u}^T$
这就完美推导出了“仅需对规模极小的矩阵 $\Phi$ 求逆即可求解 $\mathbf{p}_1$”的结论。

基于上述严密推导，真正的在线编码步骤如下（注：最终发送时需通过 $P_{col}^T$ 恢复原始变量顺序。在 $\text{GF}(2)$ 中，所有的减号 $-$ 等价于加号 $+$）：
1. **离线预计算 (Offline Precomputation)**：由于 $H$ 矩阵结构一旦确定就不再改变，常数矩阵 $\Phi = E T^{-1} B + D$ 及其逆矩阵 $\Phi^{-1}$ 可以在设备出厂前一次性预先计算并固化在存储中。这属于一次性的离线开销，不计入实时的在线编码过程。
2. **在线编码 - 计算中间变量（降复杂度的核心机密）**：如果将 $(E T^{-1}A + C)$ 作为整体预计算，它将是一个巨大的稠密矩阵，导致计算退化为 $\mathcal{O}(N^2)$。因此必须将其拆解，利用 $A, C, E, T$ 的稀疏性质，依次计算一系列中间向量：
   * **步骤 2.a**: 计算向量 $\mathbf{a}^T = A\mathbf{u}^T$。*(极度稀疏的矩阵 $A$ 乘以向量，复杂度 $\mathcal{O}(N)$)*
   * **步骤 2.b**: 计算向量 $\mathbf{b}^T = T^{-1}\mathbf{a}^T$。*(注意：并非真正对 $T$ 求逆，而是利用 $T$ 的下三角特性求解方程 $T\mathbf{b}^T = \mathbf{a}^T$，通过极其高效的**前向代入（Forward Substitution）**得出，复杂度 $\mathcal{O}(N)$)*
   * **步骤 2.c**: 计算向量 $\mathbf{c}^T = C\mathbf{u}^T$。*(稀疏乘法，复杂度 $\mathcal{O}(N)$)*
   * **步骤 2.d**: 计算向量 $\mathbf{d}^T = E\mathbf{b}^T$。*(稀疏乘法，复杂度 $\mathcal{O}(N)$)*
   * **步骤 2.e**: 得到总中间向量 $\mathbf{e}^T = \mathbf{d}^T + \mathbf{c}^T$。*(即公式推导中的 $(-E T^{-1}A + C)\mathbf{u}^T$ 部分。所有前置运算均为 $\mathcal{O}(N)$)*
3. **在线编码 - 求解 $\mathbf{p}_1$**：根据推导公式计算 $\mathbf{p}_1^T = \Phi^{-1} \mathbf{e}^T$。由于 $\Phi^{-1}$ 是维度仅为 $g \times g$ 的常数矩阵，此处操作的在线计算复杂度严格控制在 $\mathcal{O}(g^2)$ 级别。
4. **在线编码 - 求解 $\mathbf{p}_2$**：再次利用 $T$ 的下三角特性，根据方程(1)计算向量 $\mathbf{f}^T = A\mathbf{u}^T + B\mathbf{p}_1^T = \mathbf{a}^T + B\mathbf{p}_1^T$。随后求解方程 $T \mathbf{p}_2^T = \mathbf{f}^T$，通过前向代入解出 $\mathbf{p}_2$。至此，整体在线编码复杂度严格降至 $\mathcal{O}(N + g^2)$。

### 方案二：准循环 LDPC (QC-LDPC) 双对角结构 (目前最主流，如 5G NR, Wi-Fi 802.11n)
**H矩阵结构设计**：这是目前工业界的绝对主流。$H$ 矩阵不再按单个比特随机生成，而是由大小为 $Z \times Z$ 的循环置换矩阵（Circulant Permutation Matrices, CPM）或全零矩阵块拼接而成（Base Graph 展开）。更关键的是，其校验部分 $H_p$ 在设计之初就被严格构造为**双对角结构（Dual-Diagonal）**或阶梯状下三角结构。因此原矩阵直接呈现为 $H = [H_u \mid H_p]$ 的天然分块形式。
**QC-LDPC编码执行流程**：
直接利用设计好的 $H = [H_u \mid H_p]$，码字 $\mathbf{c} = [\mathbf{u}, \mathbf{p}]$。要求满足 $H\mathbf{c}^T = H_u \mathbf{u}^T + H_p \mathbf{p}^T = \mathbf{0} \pmod 2$，即 $H_p \mathbf{p}^T = H_u \mathbf{u}^T$。
1. 首先计算系统信息引发的校验和向量 $\mathbf{v} = H_u \mathbf{u}^T$。由于 $H_u$ 同样是稀疏且由循环块构成的，这一步可通过简单的移位和稀疏异或完成，复杂度 $\mathcal{O}(N)$。
2. 求解双对角方程：由于 $H_p$ 呈现双对角线全为 $1$（类似 $p_1 + p_2 = v_1$，$p_2 + p_3 = v_2 \dots$ 的差分方程结构），求解向量 $\mathbf{p}$ 变成了一个简单的累加器（Accumulator）过程：
   * $p_1$ 可通过结构中的某个环（如 5G 中的 Core Graph 特定行）单独解出。
   * 后续校验比特直接递推：$p_i = p_{i-1} + v_{i-1} \pmod 2$。
**结论**：在 QC-LDPC 架构下，**完全不需要**计算或存储稠密的生成矩阵 $G$。编码器仅需要移位寄存器和异或门，通过依次累加即可完成严格 $\mathcal{O}(N)$ 复杂度的编码工作。

### 深入解析：QC-LDPC (准循环 LDPC) 的严格代数结构

在现代通信标准（如 5G NR, Wi-Fi 802.11n/ac/ax, DVB-S2）中，QC-LDPC 因为其高度适配硬件并行处理的特性而成为绝对主流。它的校验矩阵 $H$ 具有严格的两级分层代数结构：**宏观基图（Base Graph）**与**微观循环阵（Circulant Permutation Matrices, CPM）**。

#### 1. 两级映射与提升机制 (Lifting)
QC-LDPC 的矩阵并非按单个比特随机生成，而是通过将一个小规模的**基础矩阵**（Base Matrix，或称基图）按照**提升因子**（Lifting Factor, 记为 $Z$）进行“膨胀”（Lifting）得到的。

设基图矩阵为 $H_B$，维度为 $m_b \times n_b$。提升因子为 $Z$。
最终生成的实际二元校验矩阵 $H$ 的维度为 $M \times N$，其中 $M = m_b \times Z$，$N = n_b \times Z$。

基图 $H_B$ 中的每一个元素（设为 $P_{i,j}$）都代表了一个规模为 $Z \times Z$ 的微观子矩阵：
*   **空缺（Null / -1）**：如果 $P_{i,j} = -1$（部分文献用 $\infty$ 表示），则它在实际 $H$ 矩阵中被展开为一个 $Z \times Z$ 的**全零矩阵** $\mathbf{0}_{Z \times Z}$。
*   **循环移位（Shift Value）**：如果 $P_{i,j} \ge 0$，则它被展开为一个**标准单位阵经过循环右移** $P_{i,j}$ 位后得到的 $Z \times Z$ 循环置换矩阵 $I(P_{i,j})$。
    *   例如，$P_{i,j} = 0$ 代表未移位的标准单位阵 $I_{Z \times Z}$（主对角线为 1）。
    *   例如，$P_{i,j} = 1$ 代表第一行移到第二行、最后一行移到第一行的循环右移矩阵。

由此，实际的校验矩阵 $H$ 呈现出宏观上的分块循环结构：
$H = \begin{bmatrix} I(P_{0,0}) & I(P_{0,1}) & \cdots & I(P_{0, n_b-1}) \\ I(P_{1,0}) & I(P_{1,1}) & \cdots & I(P_{1, n_b-1}) \\ \vdots & \vdots & \ddots & \vdots \\ I(P_{m_b-1, 0}) & I(P_{m_b-1, 1}) & \cdots & I(P_{m_b-1, n_b-1}) \end{bmatrix}$

#### 2. 宏观结构：系统信息与校验的天然分离
在设计基图 $H_B$ 时，标准会人为将其从列的方向上严格切分为两部分：$H_B = [H_{B,u} \mid H_{B,p}]$。
*   $H_{B,u}$ 对应**系统信息部分（Systematic Part）**，其移位值设计用于优化变量节点度分布以逼近香农极限（通常这部分列重较高，包含打孔位以提供高增益）。
*   $H_{B,p}$ 对应**校验部分（Parity Part）**，这部分被严格限制为极其特殊的结构（通常是**双对角**或**阶梯下三角**），从而**彻底消灭**了传统 LDPC 编码时的矩阵求逆与稠密乘法过程。

#### 3. 校验部分 ($H_{B,p}$) 的双对角（Dual-Diagonal）核心结构
以 IEEE 802.11n 和 5G NR 的基础结构为例，$H_{B,p}$ 最核心的 Core Parity 基图结构呈现如下形式：

$H_{B,p\_core} = \begin{bmatrix} P_{0,0} & 0 & -1 & \cdots & -1 \\ P_{1,0} & 0 & 0 & \cdots & -1 \\ -1 & -1 & 0 & \ddots & -1 \\ \vdots & \vdots & \ddots & \ddots & 0 \\ -1 & -1 & \cdots & -1 & 0 \end{bmatrix}$

在这个结构中：
1.  **阶梯双对角线**：除了第一列特殊外，主对角线及其下方的次对角线上的元素全为 $0$（代表未移位的 $Z \times Z$ 单位阵 $I$）。这在展开后形成了两条平行的单位阵对角线。
2.  **全零区域**：右上角和左下角的元素全为 $-1$（代表全零矩阵）。
3.  **闭环破缺列（第一列）**：第一列包含特定的移位值（如 $P_{0,0}, P_{1,0}$），这是为了消除双对角矩阵直接相加可能导致的秩亏损（Rank Deficiency），确保整个校验矩阵满秩。

#### 4. 为什么 QC-LDPC 能够实现严格的 $\mathcal{O}(N)$ 硬件极速编码？
当我们要满足核心校验方程 $H\mathbf{c}^T = \mathbf{0}$，即 $[H_u \mid H_p] [\mathbf{u}, \mathbf{p}]^T = \mathbf{0} \implies H_p \mathbf{p}^T = H_u \mathbf{u}^T$ 时：

1.  **计算伴随向量**：首先计算右侧的校验和向量 $\mathbf{v}^T = H_u \mathbf{u}^T$。由于 $\mathbf{u}$ 是已知信息位，且 $H_u$ 是由循环移位矩阵构成的。在硬件中，矩阵乘法 $I(P_{i,j}) \mathbf{u}^T$ 等价于**对向量 $\mathbf{u}$ 进行 $P_{i,j}$ 位的循环移位**。这完全可以通过极低功耗的移位寄存器网络和稀疏异或树在 $\mathcal{O}(N)$ 内并行计算完毕。
2.  **累加器极速求解 $\mathbf{p}$**：将求得的 $\mathbf{v}$ 代入方程 $H_p \mathbf{p}^T = \mathbf{v}^T$。我们将校验比特 $\mathbf{p}$ 按大小为 $Z$ 的块划分为 $\mathbf{p}_1, \mathbf{p}_2, \dots, \mathbf{p}_{m_b}$。由于 $H_p$ 的绝大部分是双单位阵构成的双对角结构，方程在微观上直接退化为极简的差分方程组：
    *   通过特殊的首列和末行预先消元解出第一块校验向量 $\mathbf{p}_1$。
    *   从第二块开始，后续微观方程形式全部呈现为：$\mathbf{p}_{i-1} + \mathbf{p}_{i} = \mathbf{v}_{i-1} \pmod 2$。
    *   此时，求解剩余所有校验块的过程变成了极其简单的前向**递推异或累加（Accumulation）**：$\mathbf{p}_{i} = \mathbf{p}_{i-1} + \mathbf{v}_{i-1} \pmod 2$。

**理论结论**：QC-LDPC 通过“基图设定宏观双对角结构 + 提升因子决定微观循环移位矩阵”的精妙代数设计，既保证了宏观上无需预先计算或存储任何稠密矩阵 $\Phi$ 或 $G$（实现纯粹的 $\mathcal{O}(N)$ 复杂度），又保证了微观上极度规整、避免了存储器读写冲突（Memory Conflict），完美解决了现代通信芯片在 Tbps 级吞吐量下的编码和译码并行化刚需。

## 阶段三：信道映射与传输 (Transmission)
**输入**：码字 $\mathbf{c}$。
**执行步骤**：
1. **调制映射**：将二进制码字映射为物理信道符号（以BPSK为例），执行变换 $x_i = 1 - 2c_i$。此时 $0 \to +1$，$1 \to -1$。
2. **信道衰减与加噪**：信号通过信道，接收端收到实数向量 $\mathbf{y} = [y_1, y_2, \dots, y_N]$。在AWGN信道下，$y_i = x_i + n_i$，其中 $n_i \sim \mathcal{N}(0, \sigma^2)$。
3. **计算初始对数似然比 (LLR)**：基于接收到的连续值 $y_i$ 和信道噪声方差 $\sigma^2$，计算每个变量节点 $v_j$ 的先验LLR：$L^{(0)}(v_j) = \ln \frac{P(c_j=0 | y_j)}{P(c_j=1 | y_j)} = \frac{2y_j}{\sigma^2}$

## 阶段四：置信传播 (BP) 迭代译码过程
**输入**：初始 LLR 向量 $L^{(0)}$，原始奇偶校验矩阵 $H$。
**执行步骤**：

**步骤 4.1：初始化 (Initialization)**
对于所有的 $(i, j)$ 满足 $H_{i,j} = 1$，将变量节点 $v_j$ 传递给校验节点 $c_i$ 的初始消息 $L_{v_j \to c_i}^{(0)}$ 设置为信道初始 LLR：$L_{v_j \to c_i}^{(0)} = L^{(0)}(v_j)$。设置迭代计数器 $l = 1$，最大迭代次数为 $l_{\max}$。

**步骤 4.2：校验节点更新 (Check Node Update, CNU)**
对于每个校验节点 $c_i$，根据其相连的变量节点（除目标节点 $v_j$ 外）传入的消息，计算传回给 $v_j$ 的外部信息 $L_{c_i \to v_j}^{(l)}$：$L_{c_i \to v_j}^{(l)} = 2 \tanh^{-1} \left( \prod_{j' \in \mathcal{N}(c_i) \setminus \{j\}} \tanh\left(\frac{L_{v_{j'} \to c_i}^{(l-1)}}{2}\right) \right)$ *(注：硬件实现中常用 Min-Sum 算法将其简化为符号相乘与寻找最小值的操作。)*

**步骤 4.3：变量节点更新 (Variable Node Update, VNU)**
对于每个变量节点 $v_j$，整合来自信道的先验信息与所有相连校验节点（除目标节点 $c_i$ 外）传来的外部信息，计算下一轮传递给 $c_i$ 的消息：$L_{v_j \to c_i}^{(l)} = L^{(0)}(v_j) + \sum_{i' \in \mathcal{N}(v_j) \setminus \{i\}} L_{c_{i'} \to v_j}^{(l)}$

**步骤 4.4：后验概率计算与硬判决 (Hard Decision)**
综合变量节点 $v_j$ 收到的所有信息（包括信道信息和所有校验节点信息），计算总的后验 LLR：$L_{total}(v_j) = L^{(0)}(v_j) + \sum_{i \in \mathcal{N}(v_j)} L_{c_i \to v_j}^{(l)}$。执行硬判决得到估计码字 $\hat{\mathbf{c}}$：
* 若 $L_{total}(v_j) \ge 0$，则判决 $\hat{c}_j = 0$
* 若 $L_{total}(v_j) < 0$，则判决 $\hat{c}_j = 1$

**步骤 4.5：伴随式校验与停止准则 (Syndrome Check)**
计算伴随式向量 $\mathbf{S} = \hat{\mathbf{c}}H^T \pmod 2$。
* 若 $\mathbf{S} = \mathbf{0}$，说明所有校验方程均满足，译码成功，跳出迭代。
* 若 $\mathbf{S} \neq \mathbf{0}$：
  * 若 $l < l_{\max}$，令 $l = l + 1$，返回**步骤 4.2** 继续下一次迭代。
  * 若 $l = l_{\max}$，说明达到最大迭代次数仍未收敛，声明译码失败 (Decoding Failure)。

## 阶段五：信息恢复
**输入**：译码成功的估计码字 $\hat{\mathbf{c}}$。
**执行步骤**：
将估计码字 $\hat{\mathbf{c}}$ 通过逆向的列置换操作 $\hat{\mathbf{c}}_{sys} = \hat{\mathbf{c}} \Pi$ 恢复为标准系统码字顺序，随后直接提取前 $K$ 个比特，即为恢复的原始信息 $\hat{\mathbf{u}} = [\hat{c}_{sys,1}, \hat{c}_{sys,2}, \dots, \hat{c}_{sys,K}]$。

### 附注：深入理解步骤 4.2 (校验节点更新，CNU)

步骤 4.2 是整个 LDPC 置信传播（BP）译码算法中最核心，也是数学形式最复杂的环节。我们可以从物理直觉、数学原理和工程实现三个层面来拆解这一步：

#### 1. 物理直觉：外信息 (Extrinsic Information) 传递
你可以把“校验节点”想象成一个**“会议室”**，它的唯一规则是：**所有参与这个会议的变量节点，它们的值的异或结果必须为 0（即总和为偶数）**。

当校验节点 $c_i$ 要给变量节点 $v_j$ 发送消息 $L_{c_i \to v_j}$ 时，这条消息的含义是：“根据除你之外其他所有人的表态，为了满足偶数规则，**我猜你应该是个什么值，以及我对此有多确定**。”

这就是为什么公式底下的连乘符号是 $\prod_{j' \in \mathcal{N}(c_i) \setminus \{j\}}$ —— 也就是**严格排除掉 $v_j$ 自己**。这避免了“回音壁效应”（把 $v_j$ 告诉 $c_i$ 的信息再原封不动地还给 $v_j$），防止节点产生正反馈而盲目自信，从而保证算法能够收敛。

#### 2. 数学原理：对数似然比 (LLR) 的加法同构变换
传递的信息 $L$ 是**对数似然比 (LLR)**。在概率论中，两个独立比特进行异或（模 2 加法）操作时，直接在 LLR 域计算联合概率非常复杂。但通过 $\tanh$ 函数可以实现一个极其美妙的变换（称为和积算法，Sum-Product Algorithm, SPA）：
*   **$\tanh\left(\frac{L}{2}\right)$ 映射**：将 LLR 完美地映射到 $[-1, 1]$ 的概率空间内。
*   **乘法等价**：在映射到这个空间后，**多个独立比特异或后的分布，直接等价于它们各自映射值的连乘！** 这就是公式中 $\prod$ 的由来。
*   **$2 \tanh^{-1}(\cdot)$ 反变换**：把乘出来的结果，重新拉回到 LLR 域，打包发送给 $v_j$。

#### 3. 工程实现：最小和算法 (Min-Sum Algorithm, MSA)
上述 SPA 算法在理论上极其完美，但在真实的通信芯片（如 5G 基带或 Wi-Fi 芯片）中，计算 $\tanh$ 和 $\tanh^{-1}$ 这种非线性超越函数会极其消耗硬件资源。工程师们发现，一堆数字的乘积，其最终的幅度其实**主要由这堆数字里面绝对值最小的那一个决定**（最不确定的那个节点决定了整个校验链条的可靠性下限）。

因此，硬件实现中将步骤 4.2 简化为 **Min-Sum 算法**：
1.  **符号相乘**：算出除 $v_j$ 外其他所有节点 LLR 符号的乘积（即简单的异或逻辑）。
2.  **取最小值**：找出除 $v_j$ 外所有节点 LLR 绝对值中**最小的一个**。

将这二者结合发给 $v_j$。这样就把极其复杂的超越函数连乘，变成了极度适合芯片运算的**“异或门 + 比较器”**，使得 LDPC 能够实现 Gbps 级别的超高译码吞吐量。

#### 附录：校验节点更新公式的严格数学推导 (Gallager 引理)

如前所述，BP算法在校验节点的更新本质上是计算多个独立变量模2加（异或）为0时的对数似然比。以下是这个神奇的 $\tanh$ 等式在局部独立假设下的严格推导：

**1. 概率与 LLR 的映射关系**
设二进制随机变量 $X$ 的对数似然比为 $L = \ln \frac{P(X=0)}{P(X=1)}$。
由于 $P(X=0) + P(X=1) = 1$，我们可以解出：
$P(X=0) = \frac{e^L}{1+e^L}, \quad P(X=1) = \frac{1}{1+e^L}$

现在，我们考察两者的**概率差值**：
$P(X=0) - P(X=1) = \frac{e^L - 1}{e^L + 1}$
分子分母同除以 $e^{L/2}$，得到极其关键的一步变换：
$P(X=0) - P(X=1) = \frac{e^{L/2} - e^{-L/2}}{e^{L/2} + e^{-L/2}} = \tanh\left(\frac{L}{2}\right)$
即：任何一个二进制变量“取0的概率减去取1的概率”，正好等于其 LLR 一半的双曲正切。

**2. 两个独立变量的异或 (XOR)**
设有两个独立的二进制变量 $A$ 和 $B$，令 $C = A \oplus B$。根据异或的真值表：
$P(C=0) = P(A=0)P(B=0) + P(A=1)P(B=1)$
$P(C=1) = P(A=0)P(B=1) + P(A=1)P(B=0)$

计算变量 $C$ 的概率差值：
$P(C=0) - P(C=1) = [P(A=0)P(B=0) + P(A=1)P(B=1)] - [P(A=0)P(B=1) + P(A=1)P(B=0)]$
提取公因式，进行因式分解：
$P(C=0) - P(C=1) = (P(A=0) - P(A=1)) \times (P(B=0) - P(B=1))$

**3. 推广到多变量与等式代入**
通过数学归纳法，上述结论可以推广到 $m$ 个独立变量的异或和。设 $Z = X_1 \oplus X_2 \oplus \dots \oplus X_m$：
$P(Z=0) - P(Z=1) = \prod_{i=1}^m (P(X_i=0) - P(X_i=1))$

现在，将第1步得到的 $\tanh$ 映射公式代入上式：
$\tanh\left(\frac{L(Z)}{2}\right) = \prod_{i=1}^m \tanh\left(\frac{L(X_i)}{2}\right)$

两边同时取反函数，即可得到最终异或和变量 $Z$ 的 LLR：
$L(Z) = 2 \tanh^{-1} \left( \prod_{i=1}^m \tanh\left(\frac{L(X_i)}{2}\right) \right)$

**结论与“近似性”探讨**：
将 $Z$ 视作校验节点 $c_i$ 计算出的用于传递给 $v_j$ 的“除 $v_j$ 外其他变量的和”（必须满足它等于 $v_j$ 以确保异或总和为0），我们就得到了步骤4.2中的 SPA 更新公式。**正如你所提及的，这个等式在数学上完全严格的前提是：传入校验节点的所有 $X_i$ 必须是统计独立的。** 在 LDPC 码的 Tanner 图中，只要没有成环（即局部是树状结构），这种独立性就成立。一旦图中存在环，消息会在循环中传递，变量将不再严格独立，此时公式计算出的就不再是全局真实的后验似然比（Exact Likelihood），而是变成了“带环置信传播 (Loopy BP)”的近似似然比。

#### 附录：带环置信传播 (Loopy BP) 的近似度与理论基础

在 Tanner 图存在环（Loop）的情况下，变量不再独立，BP 算法不再计算精确的全局后验概率。然而，Loopy BP 在工程中却表现出惊人的性能。理论界对 Loopy BP 的“近似度”证明主要建立在以下两个里程碑式的理论框架上：

**1. 拓扑视角：计算树展开 (Computation Tree Unrolling)**
*   **理论核心**：R. G. Gallager 和 N. Wiberg 指出，如果在有环图上运行 $l$ 次 BP 迭代，其产生的信息等价于将原图以目标节点为根，向外“展开”成一棵深度为 $l$ 的**计算树 (Computation Tree)**，然后在这棵树上进行精确的（无环）推断。
*   **近似度分析**：在计算树中，原图中的同一个节点可能会在树的不同分支和层级中重复出现多次。Loopy BP 的近似误差来源于它**将这些重复出现的同一节点当作了独立的先验输入**。当图的“围长（Girth，最短环的长度）”很大时，在早期的迭代中，计算树是严格无环的，此时 BP 的结果是**精确等于**真实后验似然的（这也是为什么 LDPC 码设计时要极力最大化围长，通常要求 Girth $\ge 6$ 甚至 $8$）。
*   **渐近精确性 (Density Evolution)**：Richardson 和 Urbanke 证明了，当码长 $N \to \infty$ 时，局部邻域在固定迭代次数 $l$ 下以概率 1 呈树状结构。此时 Loopy BP 的译码阈值（Threshold）是严格解析可证的。

**2. 统计物理视角：Bethe 自由能 (Bethe Free Energy)**
*   **理论核心**：2001年，Yedidia, Freeman 和 Weiss 发表了震撼学术界的论文，证明了 Loopy BP 并非毫无根据的启发式算法，它与统计力学中的能量近似模型有极其深刻的数学等价性。
*   **变分推断**：精确推断本质上是寻找使系统的“吉布斯自由能 (Gibbs Free Energy)”最小化的概率分布。由于有环图的吉布斯自由能无法精确求解，物理学家常使用一种叫 **Bethe 变分近似** 的方法，它只考虑单节点和节点对（边）的局部相互作用，忽略长程相关性。
*   **证明结论**：定理证明，**Loopy BP 算法的信息传递更新公式，等价于在使用拉格朗日乘子法求解 Bethe 自由能的驻点 (Stationary Points)**。并且，**Loopy BP 的任何一个收敛不动点 (Fixed Point)，都严格对应于 Bethe 自由能的一个局部极小值或鞍点**。

**总结**：
Loopy BP 的近似度并没有一个简单的绝对数值界限，而是取决于图的拓扑结构（尤其是短环的数量）和信道的噪声水平。但 Bethe 自由能理论证明了，只要 BP 能够收敛，它找到的就是在局部约束（只看单点和边）下“能量最优”的近似后验分布，这在数学上保证了它作为一种近似算法的合理性与极高的准确度。

## 代码演示：真正的朴素 LDPC 编码过程 (含 GF(2) 高斯消元)

下面的代码严格遵照了**阶段一**和**阶段二**的标准代数流程：
1. 随机生成一个较大且真正的**稀疏满秩校验矩阵** $H$。
2. 通过 **GF(2) 高斯消元法**找到 $M$ 个线性无关的列，相当于执行列置换 $\Pi$。
3. 提取出非系统部分 $P$，构造系统生成矩阵 $G_{sys} = [I_K \mid P^T]$。
4. 对信息进行系统编码后，执行**逆列置换** $\Pi^T$，将码字比特放回原始稀疏矩阵对应的列位置，完成编码闭环。

In [ ]:
import numpy as np

# =========================================
# 阶段一：系统初始化 (参数定义与GF(2)高斯消元)
# =========================================
N = 100 # 码字总长度 (Code Length)
M = 50  # 校验位长度 (Parity Length)
K = N - M # 信息位长度 (Message Length)

print(f"初始化真实稀疏 LDPC 编码演示：(N={N}, K={K})")

# 1. 随机生成稀疏满秩的校验矩阵 H
# 确保矩阵密度较低 (设定 p=0.06，使得每列/行有极少量的 1)
np.random.seed(42)
def generate_full_rank_sparse_H(M, N, p=0.06):
    while True:
        H = np.random.choice([0, 1], size=(M, N), p=[1-p, p])
        # 过滤掉全零行或全零列，稍微加速筛选
        if np.any(H.sum(axis=1) == 0) or np.any(H.sum(axis=0) == 0):
            continue

        # 使用临时矩阵进行 GF(2) 高斯消元以检查秩
        A = H.copy()
        r = 0
        for c in range(N):
            if r >= M: break
            pivot = np.argmax(A[r:, c]) + r
            if A[pivot, c] == 0: continue
            if pivot != r: A[[r, pivot]] = A[[pivot, r]]
            for i in range(M):
                if i != r and A[i, c] == 1:
                    A[i] = (A[i] + A[r]) % 2
            r += 1
        if r == M:
            return H

H = generate_full_rank_sparse_H(M, N, p=0.06)
sparsity = np.sum(H) / (M * N)
print(f"\n1. 成功生成稀疏满秩校验矩阵 H (稀疏度: {sparsity:.2%}):")
print(H)

# 2. 对 H 进行 GF(2) 高斯消元，寻找系统形式和列置换
def get_systematic_form(H, M, N):
    A = H.copy()
    pivots = [] # 记录线性无关的列索引
    r = 0
    for c in range(N):
        if r >= M: break
        pivot = np.argmax(A[r:, c]) + r
        if A[pivot, c] == 0: continue
        if pivot != r: A[[r, pivot]] = A[[pivot, r]]
        for i in range(M):
            if i != r and A[i, c] == 1:
                A[i] = (A[i] + A[r]) % 2
        pivots.append(c)
        r += 1

    non_pivots = [c for c in range(N) if c not in pivots]
    # P 矩阵就是消元后 A 矩阵中非主元列构成的子矩阵
    P = A[:, non_pivots]
    return P, pivots, non_pivots

P, pivots, non_pivots = get_systematic_form(H, M, N)
print(f"\n2. 执行GF(2)高斯消元找到 {M} 个线性无关列 (作为系统校验位)。")
print(f"   推导得到系统形式的 P 矩阵，构造 G_sys = [I_K | P^T] ...")

G_sys = np.hstack((np.eye(K, dtype=int), P.T))

# =========================================
# 阶段二：编码执行过程
# =========================================
print("\n=== 阶段二：编码执行 ===")

# 1. 随机生成待发送的信息比特向量 u
u = np.random.choice([0, 1], size=K)

# 2. 系统码编码
c_sys = np.dot(u, G_sys) % 2

# 3. 逆向列置换恢复：将生成的码字位放回原校验矩阵 H 对应的列位置
c = np.zeros(N, dtype=int)
c[non_pivots] = c_sys[:K]   # 前 K 位是信息位，放回非主元列
c[pivots] = c_sys[K:]       # 后 M 位是校验位，放回主元列

print(f"输入原始信息 u ({K} bits):\n{u}")
print(f"\n执行系统编码并应用逆列置换，生成最终码字 c ({N} bits):\n{c}")

# 4. 验证码字合法性 (伴随式校验 S = H * c^T mod 2)
syndrome = np.dot(H, c) % 2
if np.all(syndrome == 0):
    print("\n-> 验证通过：伴随式全为 0，编码合法！完美满足原稀疏矩阵 H 的正交约束。")
else:
    print("\n-> 验证失败：伴随式非零。")


初始化真实稀疏 LDPC 编码演示：(N=100, K=50)

1. 成功生成稀疏满秩校验矩阵 H (稀疏度: 6.10%):
[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [1 0 0 ... 0 0 0]]

2. 执行GF(2)高斯消元找到 50 个线性无关列 (作为系统校验位)。
   推导得到系统形式的 P 矩阵，构造 G_sys = [I_K | P^T] ...

=== 阶段二：编码执行 ===
输入原始信息 u (50 bits):
[0 0 1 1 0 0 0 1 0 0 1 1 0 0 1 1 1 1 0 0 0 1 1 0 1 0 1 1 0 0 1 0 1 1 1 1 1
 0 0 1 1 0 0 0 1 0 0 0 1 0]

执行系统编码并应用逆列置换，生成最终码字 c (100 bits):
[0 0 1 0 0 0 1 1 0 1 1 0 0 1 0 0 0 1 0 0 1 1 1 0 1 0 1 0 1 0 1 1 0 0 0 0 0
 1 1 0 0 1 0 1 1 0 1 0 1 0 0 1 0 1 0 0 1 0 0 1 1 0 0 1 1 1 1 0 0 0 1 1 1 0
 1 0 1 1 0 0 1 0 1 1 1 1 1 0 0 1 1 0 0 0 1 0 0 0 1 0]

-> 验证通过：伴随式全为 0，编码合法！完美满足原稀疏矩阵 H 的正交约束。


## 代码演示：传输与基于 Tanh 的 SPA 译码过程

下面的代码接续前面的编码结果，模拟完整的通信链路：
1. **信道映射 (BPSK)**：将 $0/1$ 映射为 $+1/-1$。
2. **信道传输 (AWGN)**：向模拟信号中加入高斯白噪声，模拟真实干扰。
3. **SPA 置信传播译码**：严格使用 $\tanh$ 和 $\tanh^{-1}$ 公式进行节点消息更新。
4. **硬判决与恢复**：通过后验 LLR 判决出码字，并通过对应的列索引恢复出原始信息比特。

In [ ]:
# =========================================
# 阶段三：信道映射与传输 (AWGN信道)
# =========================================
print("\n=== 阶段三：信道映射与传输 ===")

# 1. BPSK 调制 (0 -> +1, 1 -> -1)
x = 1 - 2 * c

# 2. 设定信噪比并生成高斯白噪声
SNR_dB = 4.0 # 设定信噪比 (dB)
SNR_linear = 10 ** (SNR_dB / 10)
R = K / N    # 编码速率
# 计算噪声标准差 sigma (Eb/N0与方差的换算)
sigma = np.sqrt(1 / (2 * R * SNR_linear))

np.random.seed(100)
noise = sigma * np.random.randn(N)
y = x + noise

print(f"设定 SNR = {SNR_dB} dB, 噪声标准差 sigma = {sigma:.4f}")
print(f"发送调制信号 x (前10位): {x[:10]}")
print(f"接收含噪信号 y (前10位): {np.round(y[:10], 2)}")

# 3. 计算接收端的初始对数似然比 (LLR)
# LLR > 0 倾向于判决为 0; LLR < 0 倾向于判决为 1
L_c = 2 * y / (sigma ** 2)

# =========================================
# 阶段四：置信传播 (BP) 迭代译码 (Sum-Product Algorithm)
# =========================================
print("\n=== 阶段四：置信传播(BP)迭代译码 ===")
max_iter = 30

# 预处理：获取 Tanner 图的连接边关系，加速遍历
# check_nodes[i] 存储连接到第 i 个校验节点的变量节点索引
check_nodes = [np.where(H[i, :] == 1)[0] for i in range(M)]
# var_nodes[j] 存储连接到第 j 个变量节点的校验节点索引
var_nodes = [np.where(H[:, j] == 1)[0] for j in range(N)]

# 初始化消息矩阵
# msg_v2c[j, i]: 变量节点 j 传给校验节点 i 的消息
# msg_c2v[i, j]: 校验节点 i 传给变量节点 j 的消息
msg_v2c = np.zeros((N, M))
msg_c2v = np.zeros((M, N))

# 初始化：V2C 消息设为初始信道 LLR
for j in range(N):
    for i in var_nodes[j]:
        msg_v2c[j, i] = L_c[j]

decoded_c = np.zeros(N, dtype=int)
success = False

for iter_idx in range(max_iter):
    # 1. 校验节点更新 (CNU): 更新 msg_c2v
    for i in range(M):
        v_connected = check_nodes[i]
        if len(v_connected) == 0: continue

        # 提取传入的消息并计算 tanh(L/2)
        incoming_msgs = msg_v2c[v_connected, i]
        tanh_msgs = np.tanh(incoming_msgs / 2.0)

        for j_idx, j in enumerate(v_connected):
            # 计算排除当前节点后的乘积
            prod = np.prod(np.delete(tanh_msgs, j_idx))
            # 极其重要：数值稳定性保护，防止乘积恰好为 1/-1 导致 arctanh 趋于无穷
            prod = np.clip(prod, -1.0 + 1e-15, 1.0 - 1e-15)
            # 计算外信息并传给变量节点
            msg_c2v[i, j] = 2 * np.arctanh(prod)

    # 2. 变量节点更新 (VNU): 更新 msg_v2c
    for j in range(N):
        c_connected = var_nodes[j]
        if len(c_connected) == 0: continue

        incoming_msgs = msg_c2v[c_connected, j]

        for i_idx, i in enumerate(c_connected):
            # 排除来自目标校验节点的消息后，将剩余消息与初始LLR相加
            sum_others = np.sum(np.delete(incoming_msgs, i_idx))
            msg_v2c[j, i] = L_c[j] + sum_others

    # 3. 后验概率计算与硬判决 (Hard Decision)
    L_total = np.zeros(N)
    for j in range(N):
        c_connected = var_nodes[j]
        L_total[j] = L_c[j] + np.sum(msg_c2v[c_connected, j])
        decoded_c[j] = 0 if L_total[j] >= 0 else 1

    # 4. 伴随式校验，若全0则提前跳出
    syndrome_check = np.dot(H, decoded_c) % 2
    if np.all(syndrome_check == 0):
        print(f"-> 译码成功！在第 {iter_idx + 1} 次迭代时伴随式收敛为全 0。")
        success = True
        break

if not success:
    print(f"-> 译码失败：达到最大迭代次数 {max_iter} 后仍未收敛。")

# =========================================
# 阶段五：信息恢复
# =========================================
print("\n=== 阶段五：信息恢复验证 ===")
bit_errors = np.sum(c != decoded_c)
print(f"接收码字总错误比特数: {bit_errors} / {N}")

# 从非主元列(非系统校验位列)恢复系统信息比特
decoded_u = np.zeros(K, dtype=int)
decoded_u = decoded_c[non_pivots]

info_errors = np.sum(u != decoded_u)
print(f"恢复原始信息错误数: {info_errors} / {K}")

if info_errors == 0:
    print("-> 验证完美：译码并正确恢复了所有的信息比特！")
else:
    print("-> 译码后信息位仍存在错误。")



=== 阶段三：信道映射与传输 ===
设定 SNR = 4.0 dB, 噪声标准差 sigma = 0.6310
发送调制信号 x (前10位): [ 1  1 -1  1  1  1 -1 -1  1 -1]
接收含噪信号 y (前10位): [-0.1   1.22 -0.27  0.84  1.62  1.32 -0.86 -1.68  0.88 -0.84]

=== 阶段四：置信传播(BP)迭代译码 ===
-> 译码成功！在第 2 次迭代时伴随式收敛为全 0。

=== 阶段五：信息恢复验证 ===
接收码字总错误比特数: 0 / 100
恢复原始信息错误数: 0 / 50
-> 验证完美：译码并正确恢复了所有的信息比特！


## 阶段二（进阶）：Richardson-Urbanke (RU) 算法的贪心三角化

传统的稀疏矩阵高斯消元会得到稠密的生成矩阵。RU 算法的核心在于**“只允许行列交换”**，试图将稀疏矩阵 $H$ 转化为近似下三角形式（ALT）：
$$
H_{ALT} = \begin{bmatrix} A_{(M-g) \times K} & B_{(M-g) \times g} & T_{(M-g) \times (M-g)} \\ C_{g \times K} & D_{g \times g} & E_{g \times (M-g)} \end{bmatrix}
$$
其中 $T$ 是一个极度稀疏的下三角方阵。由于只能通过交换操作，纯下三角很难直接达成，必定会遗留一部分“间隙” (Gap)，大小记为 $g$。

### 为什么 $g = \mathcal{O}(\sqrt{N})$？
RU 算法通过**“贪心找行重为 1”**的策略工作。由于 LDPC 的高度稀疏性（如列重通常极低，平均只有 3 左右），当我们删去一列时，与之相交的多行都会减少一个“1”，这极易产生新的“残余行重为 1”的行（类似推倒多米诺骨牌的连锁反应）。
Richardson 和 Urbanke 的理论证明指出：对于经过**精心优化的度分布**（满足特定微分方程演化条件），这种连锁反应断裂（即卡壳，没有任何行重为 1 的行）的概率极低。每一次卡壳，我们只需将一列踢入间隙区 $g$ 即可恢复连锁反应。最终，在码长 $N \to \infty$ 的系综统计下，卡壳的总次数（即间隙 $g$ 的大小）随着 $\mathcal{O}(\sqrt{N})$ 增长，远小于 $\mathcal{O}(N)$。这就保证了编码复杂度的渐近最优。

下面的代码演示了这个核心的**贪心三角化 (Greedy Triangulation)** 过程。

### RU 对 $H$ 矩阵的真实构造规则：度分布与配制模型 (Configuration Model)

Richardson 和 Urbanke 在其经典论文中提出，保证 $g = \mathcal{O}(\sqrt{N})$ 的关键不在于特定的几何拓扑，而在于**宏观统计规律**。他们构造 $H$ 矩阵的规则如下：

#### 1. 构造规则：配制模型 (Socket Model)
RU 并不像我们最开始那样“以某个概率 $p$ 抛硬币”来生成 1。抛硬币会导致行重和列重极度不均。RU 的构造规则是：
* **明确度分布**：根据优化好的度分布多项式 $\lambda(x)$ 和 $\rho(x)$，事先精确计算出：有多少个变量节点（列）度数为 2，多少个度数为 3...；有多少个校验节点（行）度数为 5，多少个度数为 6...。
* **分配插座 (Sockets)**：给每个节点分配对应数量的“插座”（即长出几根线）。
* **随机置换对接**：将所有变量节点的插座集合与所有校验节点的插座集合进行**完全随机的连线对接 (Random Permutation)**。

#### 2. 为什么这样构造能做到 $g \in \mathcal{O}(\sqrt{N})$？(核心理论)
这是一个物理学与概率论的绝妙结合，基于 **Wormald 微分方程逼近理论**：
* **流体极限 (Fluid Limit)**：在贪心三角化（不断删除行重为 1 的行）的过程中，未处理矩阵中“行重为 1 的行的数量”可以看作一个随时间演化的随机过程。当码长 $N \to \infty$ 时，这个随机过程的期望轨迹会收敛于一条确定的常微分方程（ODE）曲线。
* **保持期望为正**：RU 通过优化度分布 $(\lambda, \rho)$，确保了在整个消元过程中，这条微分方程所代表的**“行重为 1 的数量期望值”始终严格大于 0**。
* **统计涨落 (Statistical Fluctuations)**：既然期望值 $>0$，为什么还会“卡壳”（行重为 1 的数量降到 0）呢？这是因为系统存在随机性，实际数量会围绕期望值发生**统计涨落（方差）**。根据中心极限定理，这种随机游走的涨落幅度是 $\mathcal{O}(\sqrt{N})$ 级别的。
* **结论**：因为期望轨道是平稳的，系统只有在遇到极端向下涨落时才会偶然触底（卡壳）。触底时，算法“妥协”踢出一列（$g$ 增加 1），系统立刻恢复活力并重新回到期望轨道上。因此，**卡壳的总次数（即间隙 $g$）完全由统计涨落决定，其量级严格被限制在 $\mathcal{O}(\sqrt{N})$。**

### 1. 标准的 $\lambda(x)$ 和 $\rho(x)$ 定义是什么？

在 LDPC 理论（尤其是密度演化理论和 RU 架构中），度分布多项式通常是**基于边视角 (Edge-Perspective)** 的，而不是基于节点视角。

*   **变量节点度分布 $\lambda(x)$**：
    $$\lambda(x) = \sum_{d=2}^{d_{v,\max}} \lambda_d x^{d-1}$$
    其中，$\lambda_d$ 表示**所有边中，连接到度数为 $d$ 的变量节点的边所占的比例**。注意指数是 $d-1$，这是因为在置信传播（BP）译码时，沿着一条边传入节点的消息，会与另外 $d-1$ 条边传入的消息进行计算。
*   **校验节点度分布 $\rho(x)$**：
    $$\rho(x) = \sum_{d=2}^{d_{c,\max}} \rho_d x^{d-1}$$
    其中，$\rho_d$ 表示**所有边中，连接到度数为 $d$ 的校验节点的边所占的比例**。

**举个例子**：对于 Regular (3, 6) 码，由于所有变量节点度数都是 3，所有边也都连向度数为 3 的节点，所以 $\lambda_3 = 1$，多项式为 $\lambda(x) = x^2$；同理 $\rho(x) = x^5$。

### 2. 代码实现细节：插座模型 (Socket Model) 与交叉重接线 (Edge Rewiring)

在接下来的代码中，我们通过**插座模型**来物理实现这种严格的度分布配置：
1. **分配插座**：如果一个变量节点分配到度数 3，我们就给它分配 3 个“插座”（在列表中写入 3 次它的 ID）。两端分别生成庞大的插座列表。
2. **随机对接**：将校验节点的插座列表进行 `random.shuffle()` 随机打乱，然后与变量节点插座列表进行一对一的 `zip` 盲连。

**⚠️ 核心工程难点：如何处理重边 (Multi-edges)？**
在纯随机对接中，不可避免地会出现**两个节点之间被连了多条边**（即重边冲突）。在二元矩阵中，两点之间只能有 0 或 1 条边。如果遇到重边，我们**绝对不能直接丢弃**这条边！因为一旦丢弃，这两个节点的实际度数就会减 1，导致最终的矩阵实际度分布发生**退化**，彻底破坏了密度演化算出的理论最优性。

**解决方案：交叉重接线 (Edge Swapping / Rewiring)**
当代码发现即将连接的边 `(c, v)` 已经存在时，它会触发重接线机制：
* 从已成功连接的“健康边”集合中随机挑出一条边 `(c_other, v_other)`。
* 尝试**交叉连接**，即检查 `(c, v_other)` 和 `(c_other, v)` 这两条新边是否都不曾存在（不冲突）。
* 如果合法，就**拆掉**那条健康边，并**连上**这两条全新的交叉边。
* 这种“偷梁换柱”的做法完美化解了重边冲突，并且**严格保证了所有参与节点的度数绝对不发生任何改变**！

### 3. 补充：非理想 $H$ 矩阵的纯置换降维（绝对保护稀疏性）

正如我们在理论推导中所强调的，如果初始生成的稀疏校验矩阵 $H$ 存在冗余方程，计算出的核心矩阵 $\Phi$（大小为 $g \times g$）必定是奇异的（秩为 $g' < g$）。

如果直接使用代数高斯消元（例如左乘某个消元矩阵 $Q$）来强行消除冗余，**会引发灾难性的工程后果**：原本极度稀疏的校验方程相互异或，会导致新矩阵中产生大量稠密的“1”，彻底破坏了精心设计的节点度分布和 Tanner 图拓扑。

为了在**绝对不破坏任何稀疏性**的前提下解决秩亏损问题，我们必须纯粹依赖**行列置换**。

#### 核心思路：$\Phi$ 矩阵的纯置换满秩块提取
我们只需在 $g \times g$ 的秩亏损矩阵 $\Phi$ 中，寻找两个纯粹的置换矩阵（行置换 $\Pi_{left2}$ 和列置换 $\Pi_{right2}$），将 $\Phi$ 的行和列重新排列为分块形式，使得其**右上角**正好是一个 $g' \times g'$ 的满秩子矩阵 $\Phi_{core}$：
$$
\Pi_{left2} \Phi \Pi_{right2} = \begin{bmatrix} \Phi_{free (g' \times (g-g'))} & \Phi_{core (g' \times g')} \\ \Phi_{dep1} & \Phi_{dep2} \end{bmatrix}
$$
*(注：$\Phi_{core}$ 是满秩可逆的。由于整体秩仅为 $g'$，底部的 $g-g'$ 行在代数上必然是前 $g'$ 行的线性组合。)*

#### $H_{ALT}$ 的裁剪与重定义 (Pruning without Density Evolution Damage)
为什么强调要是**右上角**满秩？因为在这个分块下，接下来该干什么就极其显然且自然了：

1. **绝对安全的物理删行（不作任何加减消元）**：
   底部的 $g-g'$ 行代表了原系统中的冗余方程。我们直接利用 $\Pi_{left2}$ 记录的原始行索引，**把 $H_{ALT}$ 中这 $g-g'$ 个完整的校验行物理删除**。因为我们没有做任何方程的合并操作，所以保留下来的 $M - (g-g')$ 行**完美维持了其原始的极低行重和纯正的稀疏拓扑！**

2. **顺水推舟的列重划（左侧转信息，右侧留校验）**：
   经过 $\Pi_{right2}$ 的重排，间隙变量 $\mathbf{p}_1$ 被划分为了两部分：左侧对应 $\Phi_{free}$ 的 $g-g'$ 个自由比特，以及右侧对应 $\Phi_{core}$ 的 $g'$ 个受约束比特。
   既然物理方程只剩下 $g'$ 个，那么自然只能解出 $g'$ 个校验位。此时多余的 $g-g'$ 列恰好在**左侧**！我们只需顺水推舟，把对应 $\Phi_{free}$ 的这左半部分自由列从校验区域剥离，**直接向左融合进相邻的“系统信息列 $\mathbf{u}$”中去作为自由的信息变量**。而右侧的 $g'$ 列则安稳地留在原地，作为真正的有效间隙校验位 $\mathbf{p}_{1, core}$。

经过这一步高雅的“纯置换提取与物理裁剪”，新的间隙核心矩阵缩减为极其完美的满秩阵 $\Phi_{core}$，系统重新获得了严格的满秩属性。在这个过程中，**没有向矩阵中添加任何一个“1”，没有任何一条边被破坏**，LDPC 码的近似线性时间编码框架和理论极限的纠错性能得以 100% 完美保全。

In [22]:
import numpy as np
import random
import scipy.sparse as sp
import time

# =========================================
# 严格的 RU 架构 H 矩阵构造与极速满秩检查
# =========================================

N_ru = 10000 # 码长扩展到真实的万级规模 (N=10000)
M_ru = 5000  # Rate = 1/2

print("=== 严格 RU H 矩阵构造 (理论最优 Irregular 分布) ===")

# 1. 采用经典密度演化 (Density Evolution) 优化的渐近最优度分布 (边视角)
lambda_edges = {2: 0.2340, 3: 0.2124, 6: 0.1469, 7: 0.1128, 20: 0.2938}
rho_edges = {8: 0.7187, 9: 0.2813}

# 2. 将边比例转换为精确的节点分配数量
def generate_degree_sequence(N_nodes, edge_dist):
    norm = sum(frac / d for d, frac in edge_dist.items())
    node_counts = {d: int(np.round(N_nodes * (frac / d) / norm)) for d, frac in edge_dist.items()}

    diff = N_nodes - sum(node_counts.values())
    if diff != 0:
        max_d = max(node_counts, key=node_counts.get)
        node_counts[max_d] += diff

    sequence = []
    for d, count in node_counts.items():
        sequence.extend([d] * count)
    return sequence, node_counts

v_degrees, v_counts = generate_degree_sequence(N_ru, lambda_edges)
c_degrees, c_counts = generate_degree_sequence(M_ru, rho_edges)

# 强制边数对齐
E_v = sum(v_degrees)
E_c = sum(c_degrees)
while E_v != E_c:
    idx = random.randint(0, M_ru - 1)
    if E_v > E_c:
        c_degrees[idx] += 1
        E_c += 1
    elif E_v < E_c and c_degrees[idx] > 2:
        c_degrees[idx] -= 1
        E_c -= 1

np.random.seed(42)
np.random.shuffle(v_degrees)
np.random.shuffle(c_degrees)

print(f"码长 N={N_ru}, 校验位 M={M_ru}, 严格理论总边数 E={sum(v_degrees)}")
print(f"变量节点精确配置: 度2({v_counts.get(2,0)}个), 度3({v_counts.get(3,0)}个), 度6({v_counts.get(6,0)}个), 度7({v_counts.get(7,0)}个), 度20({v_counts.get(20,0)}个)")

def generate_strict_ru_H_sparse(N, M, v_deg, c_deg):
    start_time = time.time()
    var_sockets = []
    for j, d in enumerate(v_deg):
        var_sockets.extend([j] * d)

    chk_sockets = []
    for i, d in enumerate(c_deg):
        chk_sockets.extend([i] * d)

    random.seed(int(time.time() * 1000) % 10000)
    random.shuffle(chk_sockets)

    edges_set = set()
    edges_list = []
    pending_pairs = list(zip(chk_sockets, var_sockets))

    conflict_count = 0
    for c, v in pending_pairs:
        if (c, v) not in edges_set:
            edges_set.add((c, v))
            edges_list.append((c, v))
        else:
            conflict_count += 1
            resolved = False
            for _ in range(100):
                idx = random.randint(0, len(edges_list) - 1)
                c_other, v_other = edges_list[idx]

                if (c, v_other) not in edges_set and (c_other, v) not in edges_set:
                    edges_set.remove((c_other, v_other))
                    edges_set.add((c, v_other))
                    edges_set.add((c_other, v))

                    edges_list[idx] = (c, v_other)
                    edges_list.append((c_other, v))
                    resolved = True
                    break
            if not resolved:
                edges_set.add((c, v))
                edges_list.append((c, v))

    row_idx = [e[0] for e in edges_list]
    col_idx = [e[1] for e in edges_list]
    data = np.ones(len(edges_list), dtype=int)

    H_sparse = sp.csr_matrix((data, (row_idx, col_idx)), shape=(M, N))
    return H_sparse

def ru_greedy_triangulation(H_sparse):
    M, N = H_sparse.shape
    H_lil = H_sparse.tolil()
    row_adj = [set(cols) for cols in H_lil.rows]

    col_adj = [set() for _ in range(N)]
    for r in range(M):
        for c in row_adj[r]:
            col_adj[c].add(r)

    row_weights = np.array([len(adj) for adj in row_adj])
    weight_1_rows = set(np.where(row_weights == 1)[0])
    weight_0_rows = set(np.where(row_weights == 0)[0])

    active_rows = set(range(M))
    active_cols = set(range(N))

    order_rows_T = []
    order_rows_gap = []
    order_cols_T = []
    gap_cols = []

    while active_rows:
        if weight_1_rows:
            r = weight_1_rows.pop()
            if r not in active_rows: continue

            active_rows.remove(r)
            order_rows_T.append(r)

            c = next(iter(row_adj[r]))
            active_cols.remove(c)
            order_cols_T.append(c)

            for neighbor_r in col_adj[c]:
                if neighbor_r in active_rows:
                    row_adj[neighbor_r].remove(c)
                    w = len(row_adj[neighbor_r])
                    if w == 1:
                        weight_1_rows.add(neighbor_r)
                    elif w == 0:
                        if neighbor_r in weight_1_rows:
                            weight_1_rows.remove(neighbor_r)
                        weight_0_rows.add(neighbor_r)
        elif weight_0_rows:
            r = weight_0_rows.pop()
            if r in active_rows:
                active_rows.remove(r)
                order_rows_gap.append(r)
        else:
            best_c = -1
            max_d = -1
            for c in active_cols:
                d = sum(1 for neighbor_r in col_adj[c] if neighbor_r in active_rows)
                if d > max_d:
                    max_d = d
                    best_c = c

            if best_c != -1:
                gap_cols.append(best_c)
                active_cols.remove(best_c)

                for neighbor_r in col_adj[best_c]:
                    if neighbor_r in active_rows:
                        row_adj[neighbor_r].remove(best_c)
                        w = len(row_adj[neighbor_r])
                        if w == 1:
                            weight_1_rows.add(neighbor_r)
                        elif w == 0:
                            if neighbor_r in weight_1_rows:
                                weight_1_rows.remove(neighbor_r)
                            weight_0_rows.add(neighbor_r)
            else:
                break

    non_T_cols = gap_cols + list(active_cols)
    K = N - M
    info_cols = non_T_cols[:K]
    true_gap_cols = non_T_cols[K:]

    row_perm = order_rows_T + order_rows_gap
    col_perm = info_cols + true_gap_cols + order_cols_T

    H_csr = H_sparse.tocsr()

    A = H_csr[order_rows_T, :][:, info_cols]
    B = H_csr[order_rows_T, :][:, true_gap_cols]
    T = H_csr[order_rows_T, :][:, order_cols_T]

    C = H_csr[order_rows_gap, :][:, info_cols]
    D = H_csr[order_rows_gap, :][:, true_gap_cols]
    E = H_csr[order_rows_gap, :][:, order_cols_T]

    return row_perm, col_perm, A, B, C, D, E, T

# =========================================
# 核心运行流程：生成并执行三角化
# =========================================
t0 = time.time()
H_strict_sparse = generate_strict_ru_H_sparse(N_ru, M_ru, v_degrees, c_degrees)
print(f"-> H 矩阵生成完成 (耗时: {time.time() - t0:.3f} 秒)，开始 RU 贪心置换三角化...")

t1 = time.time()
row_perm, col_perm, A, B, C, D, E, T = ru_greedy_triangulation(H_strict_sparse)
initial_gap_size = D.shape[0]
print(f"-> 三角化完成！间隙大小 g = {initial_gap_size} (耗时: {time.time() - t1:.3f} 秒)")

# 验证 A, B, C, D, E, T 依然全部是稀疏矩阵，未被破坏！
print(f"\n子矩阵类型: A({type(A).__name__}), C({type(C).__name__}), E({type(E).__name__})")

K_ru = N_ru - M_ru
print(f"\n=== 子矩阵维度结果严格确认 ===")
print(f"间隙 g={initial_gap_size}")
print(f"A: {A.shape}, B: {B.shape}, T: {T.shape}")
print(f"C: {C.shape}, D: {D.shape}, E: {E.shape}")


=== 严格 RU H 矩阵构造 (理论最优 Irregular 分布) ===
码长 N=10000, 校验位 M=5000, 严格理论总边数 E=41128
变量节点精确配置: 度2(4813个), 度3(2913个), 度6(1007个), 度7(663个), 度20(604个)
-> H 矩阵生成完成 (耗时: 0.076 秒)，开始 RU 贪心置换三角化...
-> 三角化完成！间隙大小 g = 4 (耗时: 33.245 秒)

子矩阵类型: A(csr_matrix), C(csr_matrix), E(csr_matrix)

=== 子矩阵维度结果严格确认 ===
间隙 g=4
A: (4996, 5000), B: (4996, 4), T: (4996, 4996)
C: (4, 5000), D: (4, 4), E: (4, 4996)


In [25]:
import scipy.sparse as sp
import time
import numpy as np

# =========================================
# 1. 离线预计算：求解 Phi 并进行纯置换冗余分析
# =========================================

# 前向代入法求解 T * X = B (mod 2)
def gf2_forward_sub_matrix(T_csr, B_mat):
    n = T_csr.shape[0]
    m = B_mat.shape[1] if len(B_mat.shape) > 1 else 1
    X = np.zeros((n, m), dtype=int)
    B_dense = B_mat.toarray() if sp.issparse(B_mat) else np.atleast_2d(B_mat).reshape(n, m)

    indptr = T_csr.indptr
    indices = T_csr.indices

    for i in range(n):
        val = B_dense[i, :].copy()
        row_start = indptr[i]
        row_end = indptr[i+1]
        for j_ptr in range(row_start, row_end):
            j = indices[j_ptr]
            if j < i:
                val ^= X[j, :]
        X[i, :] = val % 2
    return X

# 新增：纯置换冗余分析核心函数 (保护稀疏性)
def analyze_phi_redundancy(Phi_dense):
    n, m = Phi_dense.shape

    # 1. 行检查：从上到下寻找冗余行
    row_basis = []
    redundant_rows = []
    indep_rows = []
    for i in range(n):
        vec = Phi_dense[i, :].copy()
        # 用现有基底进行 GF(2) 消元
        for b in row_basis:
            pivot = np.argmax(b)
            if vec[pivot] == 1:
                vec = (vec + b) % 2
        if np.all(vec == 0):
            redundant_rows.append(i)
        else:
            indep_rows.append(i)
            row_basis.append(vec)
            # 保持主元顺序，确保正确消元
            row_basis.sort(key=lambda x: np.argmax(x))

    # 2. 列检查：在剔除冗余行后，从右向左寻找自由列 (确保满秩块在右上角)
    Phi_pruned = Phi_dense[indep_rows, :]
    col_basis = []
    free_cols = []
    indep_cols = []
    for j in range(m - 1, -1, -1):
        vec = Phi_pruned[:, j].copy()
        for b in col_basis:
            pivot = np.argmax(b)
            if vec[pivot] == 1:
                vec = (vec + b) % 2
        if np.all(vec == 0):
            free_cols.append(j) # 无法独立，变为自由变量
        else:
            indep_cols.append(j)
            col_basis.append(vec)
            col_basis.sort(key=lambda x: np.argmax(x))

    # 恢复正常的从左到右索引顺序
    free_cols.reverse()
    indep_cols.reverse()

    return redundant_rows, free_cols, indep_rows, indep_cols

print("=== 离线预计算 (Offline Precomputation) ===")
start_time = time.time()
# 计算 Phi: Phi = E * T^-1 * B + D
X = gf2_forward_sub_matrix(T, B)
Phi = (E.toarray().dot(X) + D.toarray()) % 2

print(f"-> 计算得到原始 Phi 矩阵 (大小 {Phi.shape})")
redundant_rows, free_cols, indep_rows, indep_cols = analyze_phi_redundancy(Phi)

print("\n=== Phi 矩阵代数诊断结果 ===")
print(f"-> 找到冗余行 (需要物理删除): {redundant_rows}")
print(f"-> 找到自由列 (需要左移至信息位 u): {free_cols}")
print(f"-> 保留的独立行: {indep_rows}")
print(f"-> 保留的有效校验列 (构成 Phi_core): {indep_cols}")
print(f"-> 诊断耗时: {time.time() - start_time:.3f} 秒\n")

=== 离线预计算 (Offline Precomputation) ===
-> 计算得到原始 Phi 矩阵 (大小 (4, 4))

=== Phi 矩阵代数诊断结果 ===
-> 找到冗余行 (需要物理删除): [0, 1, 3]
-> 找到自由列 (需要左移至信息位 u): [0, 2, 3]
-> 保留的独立行: [2]
-> 保留的有效校验列 (构成 Phi_core): [1]
-> 诊断耗时: 0.030 秒



In [27]:
import numpy as np
import scipy.sparse as sp

print("=== 核心步骤：基于诊断结果的物理裁剪与重组 ===")

# 1. 物理删除冗余的校验行 (从 row_perm 的 gap 部分剔除)
# order_rows_T 是上部主块，order_rows_gap 是底部间隙块
order_rows_T = row_perm[:(M_ru - initial_gap_size)]
order_rows_gap = row_perm[(M_ru - initial_gap_size):]

new_order_rows_gap = [order_rows_gap[i] for i in indep_rows]
new_row_perm = order_rows_T + new_order_rows_gap

# 2. 自由列左移 (从 gap_cols 移入 info_cols)
info_cols = col_perm[:K_ru]
gap_cols = col_perm[K_ru : K_ru + initial_gap_size]
order_cols_T = col_perm[K_ru + initial_gap_size:]

free_col_indices = [gap_cols[j] for j in free_cols]
indep_col_indices = [gap_cols[j] for j in indep_cols]

new_info_cols = info_cols + free_col_indices
new_gap_cols = indep_col_indices
new_col_perm = new_info_cols + new_gap_cols + order_cols_T

# 3. 重新切分获得新的系统子矩阵
H_csr = H_strict_sparse.tocsr()

A_new = H_csr[order_rows_T, :][:, new_info_cols]
B_new = H_csr[order_rows_T, :][:, new_gap_cols]
T_new = H_csr[order_rows_T, :][:, order_cols_T]

C_new = H_csr[new_order_rows_gap, :][:, new_info_cols]
D_new = H_csr[new_order_rows_gap, :][:, new_gap_cols]
E_new = H_csr[new_order_rows_gap, :][:, order_cols_T]

# 新增：定义原始意义上的新稀疏矩阵 H_new (即仅物理删除冗余行的原矩阵，完美保持原列拓扑)
H_new = H_csr[new_row_perm, :]

# 拼合近似下三角阵 H_ALT_new (用于后续的 RU 快速编码)
H_ALT_new = sp.bmat([[A_new, B_new, T_new], [C_new, D_new, E_new]], format='csr')

# 4. 统计新维度
g_new = len(new_gap_cols)
K_new = len(new_info_cols)
M_new = len(new_row_perm)

print(f"-> 纯置换与裁剪完成！")
print(f"-> 新的信息位长度 K': {K_ru} + {len(free_cols)} = {K_new}")
print(f"-> 新的校验方程数 M': {M_ru} - {len(redundant_rows)} = {M_new}")
print(f"-> 新的核心间隙大小 g': {g_new}")
print("\n=== 新子矩阵维度确认 ===")
print(f"A_new: {A_new.shape}, B_new: {B_new.shape}, T_new: {T_new.shape}")
print(f"C_new: {C_new.shape}, D_new: {D_new.shape}, E_new: {E_new.shape}")
print(f"H_new (原始拓扑裁剪版): {H_new.shape}, 非零元素个数: {H_new.nnz}")
print(f"H_ALT_new (下三角编码版): {H_ALT_new.shape}, 非零元素个数: {H_ALT_new.nnz}")

# 5. 重新计算新的 Phi_core 并验证其满秩性
print("\n=== 终极验证：新的 Phi_core 矩阵 ===")
# 需自带一个简单的 GF(2) 求逆函数用于验证
def gf2_invert_core(matrix):
    n = matrix.shape[0]
    A_mat = np.hstack([matrix, np.eye(n, dtype=int)])
    for i in range(n):
        pivot = np.argmax(A_mat[i:, i]) + i
        if A_mat[pivot, i] == 0:
            raise ValueError("依然存在奇异性！")
        if pivot != i:
            A_mat[[i, pivot]] = A_mat[[pivot, i]]
        for j in range(n):
            if i != j and A_mat[j, i] == 1:
                A_mat[j] = (A_mat[j] + A_mat[i]) % 2
    return A_mat[:, n:]

X_new = gf2_forward_sub_matrix(T_new, B_new)
Phi_core = (E_new.toarray().dot(X_new) + D_new.toarray()) % 2
print(f"-> 提取的 Phi_core 矩阵大小: {Phi_core.shape}")

try:
    Phi_core_inv = gf2_invert_core(Phi_core)
    print("-> 完美！经过纯置换与裁剪，Phi_core 矩阵现已严格满秩，求逆成功！")
    print("-> 稀疏性 100% 保持，无任何破坏。系统现已准备好进行 O(N) 在线编码。")
except ValueError as e:
    print("-> 错误：Phi_core 依然奇异！", e)


=== 核心步骤：基于诊断结果的物理裁剪与重组 ===
-> 纯置换与裁剪完成！
-> 新的信息位长度 K': 5000 + 3 = 5003
-> 新的校验方程数 M': 5000 - 3 = 4997
-> 新的核心间隙大小 g': 1

=== 新子矩阵维度确认 ===
A_new: (4996, 5003), B_new: (4996, 1), T_new: (4996, 4996)
C_new: (1, 5003), D_new: (1, 1), E_new: (1, 4996)
H_new (原始拓扑裁剪版): (4997, 10000), 非零元素个数: 41103
H_ALT_new (下三角编码版): (4997, 10000), 非零元素个数: 41103

=== 终极验证：新的 Phi_core 矩阵 ===
-> 提取的 Phi_core 矩阵大小: (1, 1)
-> 完美！经过纯置换与裁剪，Phi_core 矩阵现已严格满秩，求逆成功！
-> 稀疏性 100% 保持，无任何破坏。系统现已准备好进行 O(N) 在线编码。


In [28]:
import numpy as np
import time

print("=== 阶段二（终极）：O(N) 极速在线编码 (RU 算法) ===")

# 1. 生成随机的原始信息向量 u (长度为新的 K_new)
np.random.seed(123)
u_new = np.random.choice([0, 1], size=K_new)

start_enc = time.time()

# ==========================================
# 2. 在线编码 - 计算中间变量 (全是极度稀疏的 O(N) 乘法和前向代入)
# ==========================================
# 步骤 2.a: a^T = A_new * u^T
a = A_new.dot(u_new) % 2

# 步骤 2.b: T_new * b^T = a^T -> 利用下三角特性前向代入求解 b
b = gf2_forward_sub_matrix(T_new, a).flatten()

# 步骤 2.c: c^T = C_new * u^T
c_vec = C_new.dot(u_new) % 2

# 步骤 2.d: d^T = E_new * b^T
d_vec = E_new.dot(b) % 2

# 步骤 2.e: e^T = c^T + d^T
e_vec = (c_vec + d_vec) % 2

# ==========================================
# 3. 在线编码 - 求解 p1 和 p2
# ==========================================
# 求解 p1 (核心 Gap 校验位): p1^T = Phi_core_inv * e^T
p1 = Phi_core_inv.dot(e_vec) % 2

# 求解 p2 (剩余校验位): f^T = a^T + B_new * p1^T
f = (a + B_new.dot(p1)) % 2
# T_new * p2^T = f^T -> 再次前向代入
p2 = gf2_forward_sub_matrix(T_new, f).flatten()

enc_time = time.time() - start_enc

# ==========================================
# 4. 组装与逆向重排
# ==========================================
# 组合置换后的码字 c_alt = [u, p1, p2]
c_alt = np.concatenate([u_new, p1, p2])

# 寻找 new_col_perm 的逆置换矩阵，将所有比特放回原矩阵对应的列位置
inverse_col_perm = np.argsort(new_col_perm)
c_final = c_alt[inverse_col_perm]

print(f"-> 编码完成！在线计算耗时: {enc_time:.5f} 秒")
print(f"-> 生成的最终码字 c_final 长度: {len(c_final)}")

# ==========================================
# 5. 终极验证：H_new * c_final^T == 0
# ==========================================
print("\n=== 终极验证：编码合法性 ===")
syndrome_final = H_new.dot(c_final) % 2

if np.all(syndrome_final == 0):
    print("-> 验证完美通过：生成的码字完全满足新的稀疏校验矩阵 H_new！")
    print("-> ✅ 物理删行 + 纯置换降阶的 O(N) 线性时间编码架构构建大获成功！")
else:
    print("-> ❌ 验证失败：存在不满足的校验方程。")


=== 阶段二（终极）：O(N) 极速在线编码 (RU 算法) ===
-> 编码完成！在线计算耗时: 0.10273 秒
-> 生成的最终码字 c_final 长度: 10000

=== 终极验证：编码合法性 ===
-> 验证完美通过：生成的码字完全满足新的稀疏校验矩阵 H_new！
-> ✅ 物理删行 + 纯置换降阶的 O(N) 线性时间编码架构构建大获成功！


In [32]:
import numpy as np
import time

print("=== 阶段三：大尺度信道映射与传输 (AWGN) ===")
# 1. BPSK 调制 (0 -> +1, 1 -> -1)
x_tx = 1 - 2 * c_final

# 2. 设定 SNR 和生成 AWGN 噪声
SNR_dB_ru = 1.2  # 设定一个考验译码性能的较低信噪比
SNR_linear_ru = 10 ** (SNR_dB_ru / 10)
R_ru = K_new / len(c_final) # 实际编码速率
sigma_ru = np.sqrt(1 / (2 * R_ru * SNR_linear_ru))

np.random.seed(42)
noise_ru = sigma_ru * np.random.randn(len(c_final))
y_rx = x_tx + noise_ru

print(f"-> 传输参数: 码长={len(c_final)}, 码率 R={R_ru:.4f}, SNR={SNR_dB_ru}dB, 噪声标准差={sigma_ru:.4f}")

# 3. 初始信道对数似然比 (LLR)
L_c_ru = 2 * y_rx / (sigma_ru ** 2)

print("\n=== 阶段四：基于 Tanh 的大尺度稀疏 BP 迭代译码 ===")
max_iter_ru = 25
M_new, N_new = H_new.shape

start_dec = time.time()

# ==========================================
# 预处理：建立高效的边级(Edge-level)索引映射
# ==========================================
# 为了在大规模矩阵中实现极速迭代，我们不使用 N*M 的庞大稠密矩阵，
# 而是仅仅跟踪 H_new 中非零的 E 条边。
rows, cols = H_new.nonzero()
E = len(rows)

# chk_edges[i] 存储连接到校验节点 i 的边在 E 数组中的索引
chk_edges = [np.where(rows == i)[0] for i in range(M_new)]
# var_edges[j] 存储连接到变量节点 j 的边在 E 数组中的索引
var_edges = [np.where(cols == j)[0] for j in range(N_new)]

# 消息数组 (长度为边数 E，极度节省内存且完全避免无效 0 计算)
msg_v2c = np.zeros(E)
msg_c2v = np.zeros(E)

# 初始化：V2C 消息设为信道初始 LLR
for j in range(N_new):
    msg_v2c[var_edges[j]] = L_c_ru[j]

decoded_c_ru = np.zeros(N_new, dtype=int)
success_ru = False

for iter_idx in range(max_iter_ru):
    # 1. 校验节点更新 (CNU)
    for i in range(M_new):
        edges = chk_edges[i]
        if len(edges) == 0: continue

        incoming = msg_v2c[edges]
        tanh_msgs = np.tanh(incoming / 2.0)

        for e_idx, e in enumerate(edges):
            # 排除当前节点，计算剩余节点的乘积
            prod = np.prod(np.delete(tanh_msgs, e_idx))
            # 极其重要：数值稳定性保护，防止 arctanh(1/-1) 无穷大
            prod = np.clip(prod, -1.0 + 1e-15, 1.0 - 1e-15)
            msg_c2v[e] = 2 * np.arctanh(prod)

    # 2. 变量节点更新 (VNU)
    for j in range(N_new):
        edges = var_edges[j]
        if len(edges) == 0: continue

        incoming = msg_c2v[edges]
        # 加上信道先验 LLR 和所有传入的校验节点消息
        total_sum = L_c_ru[j] + np.sum(incoming)

        for e_idx, e in enumerate(edges):
            # 传出的消息等于总和减去该目标校验节点传来的那份
            msg_v2c[e] = total_sum - incoming[e_idx]

    # 3. 后验概率计算与硬判决
    L_total = np.zeros(N_new)
    for j in range(N_new):
        edges = var_edges[j]
        L_total[j] = L_c_ru[j] + np.sum(msg_c2v[edges])
        decoded_c_ru[j] = 0 if L_total[j] >= 0 else 1

    # 4. 快速伴随式校验
    syndrome = H_new.dot(decoded_c_ru) % 2
    if np.all(syndrome == 0):
        print(f"-> 译码成功！在第 {iter_idx + 1} 次迭代时伴随式收敛为全 0。")
        success_ru = True
        break

dec_time = time.time() - start_dec
if not success_ru:
    print(f"-> 译码失败：达到最大迭代次数 {max_iter_ru} 后仍未收敛。")
print(f"-> BP 译码总耗时: {dec_time:.3f} 秒")

# ==========================================
# 阶段五：信息恢复与闭环验证
# ==========================================
print("\n=== 阶段五：大规模信息恢复验证 ===")
bit_errs = np.sum(c_final != decoded_c_ru)
print(f"接收码字总错误比特数 (译码硬判决后): {bit_errs} / {N_new}")

# 根据前面的组装逻辑：c_final = c_alt[inverse_col_perm]
# 因此，恢复置换顺序：c_alt = decoded_c_ru[new_col_perm]
decoded_c_alt = decoded_c_ru[new_col_perm]

# 前 K_new 位即为系统信息位 u_new
decoded_u_new = decoded_c_alt[:K_new]

info_errs = np.sum(u_new != decoded_u_new)
print(f"恢复原始信息错误数: {info_errs} / {K_new}")

if info_errs == 0:
    print("-> 验证完美：大规模 LDPC 系统 纯置换降阶编译码 全链路闭环大获成功！🎉")
else:
    print("-> 译码后信息位仍存在错误。")


=== 阶段三：大尺度信道映射与传输 (AWGN) ===
-> 传输参数: 码长=10000, 码率 R=0.5003, SNR=1.2dB, 噪声标准差=0.8707

=== 阶段四：基于 Tanh 的大尺度稀疏 BP 迭代译码 ===
-> 译码成功！在第 24 次迭代时伴随式收敛为全 0。
-> BP 译码总耗时: 22.175 秒

=== 阶段五：大规模信息恢复验证 ===
接收码字总错误比特数 (译码硬判决后): 0 / 10000
恢复原始信息错误数: 0 / 5003
-> 验证完美：大规模 LDPC 系统 纯置换降阶编译码 全链路闭环大获成功！🎉
